In [1]:
from agent_framework import Agent, tool, FunctionInvocationContext
from pydantic import Field
from typing import Annotated
from agent_framework.openai import OpenAIChatCompletionClient  # for older models (e.g., oss-120b)
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# Tool without approval mode
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str | int, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str | int:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

# Batch Requests (Chat Completion)

### HuggingFace Model

In [13]:
chat_client = OpenAIChatCompletionClient(
    base_url=os.getenv("HF_API_BASE_URL"),
    api_key=os.getenv("HF_API_KEY"),
    model="google/gemma-4-31B-it:deepinfra",
)

In [14]:
agent = Agent(
    client=chat_client,
    name="WeatherAgent",
    description="A helpful weather assistant that can answer questions about the weather.",
    instructions=(
        "You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. "
    ),
    tools=[get_weather],
    context_providers=[],
)

In [15]:
result = await agent.run("What is the weather in Halifax?")

C:\Users\JOSHUAS\AppData\Local\Temp\ipykernel_34228\110213656.py:1: RuntimeWarning: coroutine 'AgentTelemetryLayer._trace_agent_invocation.<locals>._run' was never awaited
  result = await agent.run("What is the weather in Halifax?")


In [ ]:
print(result) # prints the concatenated result of the agent's response

The weather in Halifax is cloudy with a high of 15°C.


In [20]:
for message in result.messages:
    print(message.to_dict())

{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'function_call', 'call_id': 'chatcmpl-tool-af9ad3aab301e494', 'name': 'get_weather', 'arguments': '{"location": "Halifax"}', 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'tool', 'contents': [{'type': 'function_result', 'call_id': 'chatcmpl-tool-af9ad3aab301e494', 'result': 'The weather in Halifax is cloudy with a high of 15°C.', 'items': [{'type': 'text', 'text': 'The weather in Halifax is cloudy with a high of 15°C.', 'additional_properties': {}}], 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'text', 'text': 'The weather in Halifax is cloudy with a high of 15°C.', 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}


### Foundry Model (OpenAI)

In [23]:
chat_client = OpenAIChatCompletionClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

In [24]:
agent = Agent(
    client=chat_client,
    name="WeatherAgent",
    description="A helpful weather assistant that can answer questions about the weather.",
    instructions=(
        "You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. "
    ),
    tools=[get_weather],
    context_providers=[],
)

In [25]:
result = await agent.run('What is the weather in Halifax?')

In [26]:
for message in result.messages:
    print(message.to_dict())

{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'function_call', 'call_id': 'call_h9hGEHjgOQHaTQWWdm4azMq2', 'name': 'get_weather', 'arguments': '{"location":"Halifax"}', 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'tool', 'contents': [{'type': 'function_result', 'call_id': 'call_h9hGEHjgOQHaTQWWdm4azMq2', 'result': 'The weather in Halifax is cloudy with a high of 15°C.', 'items': [{'type': 'text', 'text': 'The weather in Halifax is cloudy with a high of 15°C.', 'additional_properties': {}}], 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'text', 'text': 'The weather in Halifax is currently cloudy with a high of 15°C.', 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}


### Gemini Model (Google)

In [13]:
from agent_framework.gemini import GeminiChatClient

chat_client = GeminiChatClient(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model='gemini-3.6-flash',
)

In [14]:
agent = Agent(
    client=chat_client,
    name="WeatherAgent",
    description="A helpful weather assistant that can answer questions about the weather.",
    instructions=(
        "You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. "
    ),
    tools=[get_weather],
    context_providers=[],
)

In [15]:
result = await agent.run('What is the weather in Halifax?')
for message in result.messages:
    print(message.to_dict())

{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'text_reasoning', 'protected_data': 'EuwBCukBAWkUfRPEnb+nsLq15cD3EH76U45EC3p5Lx/HCzJh5ARxkzpTpi6TlDxMscagGDCk0e9qt7GBL3+gxW1YeDv/VPLUOPHWLvMvEu2QobbZ63Ifrvau6bTQsmenFwBBrYdV1wswSobVpLAcleYCe20sS0C/rcu+3x8CM42l7P31A5w6CLbWzCh7aO//nJdREfjnTInfbQVWVaSbJ20jMaBSO9UUlGGez7msGkr/s40p6Ouh7FoO3YjmTvJt+LkbbQ2OpBk1mQWbkQEen8/tNejEE3iObZKuxQXv/2rFhtitAkkNf8hYybbqvSc=', 'additional_properties': {}}, {'type': 'function_call', 'call_id': 'call_131430', 'name': 'get_weather', 'arguments': {'location': 'Halifax'}, 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'tool', 'contents': [{'type': 'function_result', 'call_id': 'call_131430', 'result': 'The weather in Halifax is cloudy with a high of 15°C.', 'items': [{'type': 'text', 'text': 'The weather in Halifax is cloudy with a high of 15°C.', 'additional_properties': {}}], 'additional_properties': {}}], 'author_name': 'Wea

### Local Model (Ollama)

In [7]:
from agent_framework.ollama import OllamaChatClient

chat_client = OllamaChatClient(
    # model=os.getenv("OLLAMA_MODEL"),
    model="granite4:tiny-h",
)

In [8]:
agent = Agent(
    client=chat_client,
    name="WeatherAgent",
    description="A helpful weather assistant that can answer questions about the weather.",
    instructions=(
        "You are a helpful weather assistant with access to a get_weather tool for answering questions about the weather. "
    ),
    tools=[get_weather],
    context_providers=[],
)

In [9]:
result = await agent.run('What is the weather in Halifax?')
for message in result.messages:
    print(message.to_dict())

{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'function_call', 'call_id': '3444409b-5882-474d-8436-6339ef656c5d', 'name': 'get_weather', 'arguments': {'location': 'Halifax'}, 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'tool', 'contents': [{'type': 'function_result', 'call_id': '3444409b-5882-474d-8436-6339ef656c5d', 'result': 'The weather in Halifax is cloudy with a high of 15°C.', 'items': [{'type': 'text', 'text': 'The weather in Halifax is cloudy with a high of 15°C.', 'additional_properties': {}}], 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'function_call', 'call_id': '2fd90a3e-a42d-4599-a1f2-bf55eaab4ebb', 'name': 'get_weather', 'arguments': {'location': 'Halifax'}, 'additional_properties': {}}], 'author_name': 'WeatherAgent', 'additional_properties': {}}
{'type': 'message', 'role